In [1]:

!nvidia-smi


Mon Nov 23 06:36:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

!pip install -q onnx onnxruntime



In [3]:

import torch
import onnx
import onnxruntime as ort

print("PyTorch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("ONNX Runtime:", ort.__version__)
print("GPU Available:", torch.cuda.is_available())


PyTorch: 2.14.1+cu130
ONNX: 1.23.2
ONNX Runtime: 1.32.0
GPU Available: True


In [4]:

import torch
import torch.nn as nn

# Define model architecture
class MyModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


# Initialize model
model = MyModel()

print(model)
print("\nModel architecture created successfully!")


MyModel(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

Model architecture created successfully!


In [5]:

import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Load MNIST dataset
transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Training on:", device)

# Train model
for epoch in range(3):

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/3 | "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )

# Save checkpoint
torch.save(
    model.cpu().state_dict(),
    "model.pth"
)

print("\nTrained checkpoint saved: model.pth")


100%|██████████| 9.91M/9.91M [00:00<00:00, 13.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 408kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.73MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.1MB/s]


Training on: cuda
Epoch 1/3 | Loss: 0.3384
Epoch 2/3 | Loss: 0.1554
Epoch 3/3 | Loss: 0.1069

Trained checkpoint saved: model.pth


In [6]:

import torch
import onnx

# Recreate and load trained model
model = MyModel()

model.load_state_dict(
    torch.load("model.pth", map_location="cpu")
)

model.eval()

# Dummy input
dummy_input = torch.randn(1, 1, 28, 28)

# Export to ONNX
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    dynamo=False
)

# Validate ONNX
onnx_model = onnx.load("model.onnx")
onnx.checker.check_model(onnx_model)

print("ONNX export successful!")
print("Saved file: model.onnx")


ONNX export successful!
Saved file: model.onnx


/tmp/ipykernel_540/2507587528.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [21]:

import onnxruntime as ort
import numpy as np

# Load ONNX model
session = ort.InferenceSession(
    "model.onnx",
    providers=["CPUExecutionProvider"]
)

# Generate test input
test_input = np.random.randn(
    1, 1, 28, 28
).astype(np.float32)

# Run inference
result = session.run(
    ["output"],
    {"input": test_input}
)

print("ONNX inference successful!")
print("Output shape:", result[0].shape)
print("Predicted digit:", np.argmax(result[0]))


ONNX inference successful!
Output shape: (1, 10)
Predicted digit: 0


In [22]:
print("PyTorch CUDA version:", torch.version.cuda)
!nvidia-smi

PyTorch CUDA version: 13.0
Mon Nov 23 06:40:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P0             26W /   70W |     163MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------

In [24]:
!pip install -q tensorrt-cu12

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [25]:
import tensorrt as trt

print("TensorRT version:", trt.__version__)

logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)

print("TensorRT Builder initialized successfully!")

TensorRT version: 11.5.0.63
TensorRT Builder initialized successfully!


In [33]:

import tensorrt as trt

print("TensorRT Version:", trt.__version__)

# Initialize TensorRT
logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)

# Create strongly typed network
network = builder.create_network()

# Create ONNX parser
parser = trt.OnnxParser(network, logger)

# Load ONNX model
with open("model.onnx", "rb") as f:
    onnx_data = f.read()

# Parse ONNX
if not parser.parse(onnx_data):

    print("ONNX parsing failed!")

    for i in range(parser.num_errors):
        print(parser.get_error(i))

    raise RuntimeError("ONNX parsing failed")

print("ONNX parsed successfully!")

# Configure engine
config = builder.create_builder_config()

# Set workspace memory to 1 GB
config.set_memory_pool_limit(
    trt.MemoryPoolType.WORKSPACE,
    1 << 30
)

print("Building TensorRT engine...")

# Build engine (FP32 from the FP32 ONNX model)
serialized_engine = builder.build_serialized_network(
    network,
    config
)

if serialized_engine is None:
    raise RuntimeError("TensorRT engine build failed!")

# Save engine
with open("model.engine", "wb") as f:
    f.write(serialized_engine)

print("\nTensorRT Engine created successfully!")
print("Saved file: model.engine")


TensorRT Version: 11.5.0.63
ONNX parsed successfully!
Building TensorRT engine...

TensorRT Engine created successfully!
Saved file: model.engine


In [34]:

import tensorrt as trt

logger = trt.Logger(trt.Logger.WARNING)

# Load engine file
with open("model.engine", "rb") as f:
    engine_data = f.read()

# Deserialize engine
runtime = trt.Runtime(logger)

engine = runtime.deserialize_cuda_engine(
    engine_data
)

if engine is None:
    raise RuntimeError("Engine loading failed!")

print("TensorRT Engine loaded successfully!")
print("Number of I/O tensors:", engine.num_io_tensors)

for i in range(engine.num_io_tensors):
    name = engine.get_tensor_name(i)
    print(
        name,
        engine.get_tensor_dtype(name),
        engine.get_tensor_shape(name)
    )


TensorRT Engine loaded successfully!
Number of I/O tensors: 2
input DataType.FLOAT (1, 1, 28, 28)
output DataType.FLOAT (1, 10)


In [35]:

import os
import zipfile
from google.colab import files

artifacts = [
    "model.pth",
    "model.onnx",
    "model.engine"
]

# Create ZIP file
with zipfile.ZipFile(
    "Model_Artifacts.zip", "w"
) as zipf:

    for filename in artifacts:
        if os.path.exists(filename):
            zipf.write(filename)

print("All available artifacts:")

for filename in artifacts:
    print(filename, os.path.exists(filename))

# Download ZIP
files.download("Model_Artifacts.zip")


All available artifacts:
model.pth True
model.onnx True
model.engine True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>